# Hadoop Streaming Practice in Google Colab

This notebook uses a **real Hadoop environment** in Colab and **does not use PySpark**.

It fixes the earlier issues:
- correct Hadoop download and install
- correct `PATH` / `JAVA_HOME` / `HADOOP_HOME`
- correct **streaming JAR path**
- uses **Python `subprocess`** instead of fragile notebook shell magics
- uses **local file system mode** with `file:///...`
- reads `part-00000` **only after** the Hadoop job succeeds

You do **not** need any parquet file for this practice. Hadoop Streaming is designed for **plain text** input.


In [ ]:

# Step 1: Install Java + Hadoop correctly
import os
import subprocess
from pathlib import Path

HADOOP_VERSION = "3.3.6"
HADOOP_DIR = f"/content/hadoop-{HADOOP_VERSION}"
HADOOP_TAR = f"/content/hadoop-{HADOOP_VERSION}.tar.gz"
HADOOP_URL = f"https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VERSION}/hadoop-{HADOOP_VERSION}.tar.gz"

def run(cmd, env=None):
    print("RUN:", cmd)
    completed = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {cmd}")
    return completed

run("apt-get -qq update")
run("apt-get install -y openjdk-11-jdk-headless wget tar > /dev/null")

if not Path(HADOOP_DIR).exists():
    run(f"wget -q -O {HADOOP_TAR} {HADOOP_URL}")
    run(f"tar -xzf {HADOOP_TAR} -C /content/")

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["HADOOP_HOME"] = HADOOP_DIR
os.environ["HADOOP_CONF_DIR"] = f"{HADOOP_DIR}/etc/hadoop"
os.environ["PATH"] = os.environ["PATH"] + f":{HADOOP_DIR}/bin:{HADOOP_DIR}/sbin"

ENV = os.environ.copy()

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])

run("java -version", env=ENV)
run(f"{HADOOP_DIR}/bin/hadoop version", env=ENV)


RUN: apt-get -qq update
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

RUN: apt-get install -y openjdk-11-jdk-headless wget tar > /dev/null

RUN: wget -q -O /content/hadoop-3.3.6.tar.gz https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz

RUN: tar -xzf /content/hadoop-3.3.6.tar.gz -C /content/

JAVA_HOME = /usr/lib/jvm/java-11-openjdk-amd64
HADOOP_HOME = /content/hadoop-3.3.6
RUN: java -version
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)

RUN: /content/hadoop-3.3.6/bin/hadoop version
Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with p

CompletedProcess(args='/content/hadoop-3.3.6/bin/hadoop version', returncode=0, stdout='Hadoop 3.3.6\nSource code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c\nCompiled by ubuntu on 2023-06-18T08:22Z\nCompiled on platform linux-x86_64\nCompiled with protoc 3.7.1\nFrom source with checksum 5652179ad55f76cb287d9c633bb53bbd\nThis command was run using /content/hadoop-3.3.6/share/hadoop/common/hadoop-common-3.3.6.jar\n')

In [ ]:
# Step 2: Create sample input data
from pathlib import Path

sample_dir = Path("/content/hadoop_input")
sample_dir.mkdir(exist_ok=True)

sample_text_1 = '''2026-04-22 09:36:39 INFO BlockStateChange replication started successfully
2026-04-22 09:37:10 WARN DataNode slow response from storage subsystem
2026-04-22 09:37:55 ERROR NameNode failed to process metadata synchronization
2026-04-22 09:38:15 INFO ReplicationManager completed rebalancing operation
2026-04-22 09:38:45 INFO authenticationservice accepted token request
'''

sample_text_2 = '''2026-04-22 10:01:03 INFO BlockManager received heartbeat from worker node
2026-04-22 10:02:28 WARN CheckpointCoordinator detected delayed checkpoint creation
2026-04-22 10:03:44 INFO metadata synchronization finished without exception
2026-04-22 10:04:59 ERROR configurationvalidationservice rejected invalid property
2026-04-22 10:05:31 INFO replicationmonitor completed verification successfully
'''

(sample_dir / "log1.txt").write_text(sample_text_1)
(sample_dir / "log2.txt").write_text(sample_text_2)

run("ls -R /content/hadoop_input")
run('echo "----- log1.txt -----" && cat /content/hadoop_input/log1.txt')
run('echo "----- log2.txt -----" && cat /content/hadoop_input/log2.txt')


RUN: ls -R /content/hadoop_input
/content/hadoop_input:
log1.txt
log2.txt

RUN: echo "----- log1.txt -----" && cat /content/hadoop_input/log1.txt
----- log1.txt -----
2026-04-22 09:36:39 INFO BlockStateChange replication started successfully
2026-04-22 09:37:10 WARN DataNode slow response from storage subsystem
2026-04-22 09:37:55 ERROR NameNode failed to process metadata synchronization
2026-04-22 09:38:15 INFO ReplicationManager completed rebalancing operation
2026-04-22 09:38:45 INFO authenticationservice accepted token request

RUN: echo "----- log2.txt -----" && cat /content/hadoop_input/log2.txt
----- log2.txt -----
2026-04-22 10:01:03 INFO BlockManager received heartbeat from worker node
2026-04-22 10:02:28 WARN CheckpointCoordinator detected delayed checkpoint creation
2026-04-22 10:03:44 INFO metadata synchronization finished without exception
2026-04-22 10:04:59 ERROR configurationvalidationservice rejected invalid property
2026-04-22 10:05:31 INFO replicationmonitor complete

CompletedProcess(args='echo "----- log2.txt -----" && cat /content/hadoop_input/log2.txt', returncode=0, stdout='----- log2.txt -----\n2026-04-22 10:01:03 INFO BlockManager received heartbeat from worker node\n2026-04-22 10:02:28 WARN CheckpointCoordinator detected delayed checkpoint creation\n2026-04-22 10:03:44 INFO metadata synchronization finished without exception\n2026-04-22 10:04:59 ERROR configurationvalidationservice rejected invalid property\n2026-04-22 10:05:31 INFO replicationmonitor completed verification successfully\n')

In [ ]:

# Step 3: Locate Hadoop example and streaming JAR files
import glob

example_matches = glob.glob(
    f"{os.environ['HADOOP_HOME']}/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar"
)
streaming_matches = glob.glob(
    f"{os.environ['HADOOP_HOME']}/share/hadoop/tools/lib/hadoop-streaming-*.jar"
)

assert example_matches, "Could not find Hadoop examples JAR."
assert streaming_matches, "Could not find Hadoop streaming JAR."

EXAMPLE_JAR = example_matches[0]
STREAMING_JAR = streaming_matches[0]

print("Example JAR  :", EXAMPLE_JAR)
print("Streaming JAR:", STREAMING_JAR)


Example JAR  : /content/hadoop-3.3.6/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.3.6.jar
Streaming JAR: /content/hadoop-3.3.6/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar


In [ ]:

# Step 4: Run a built-in Hadoop example (grep) in LOCAL mode
run("rm -rf /content/hadoop_grep_output")

grep_cmd = (
    f'"{os.environ["HADOOP_HOME"]}/bin/hadoop" jar "{EXAMPLE_JAR}" grep '
    f'-D mapreduce.framework.name=local '
    f'-D fs.defaultFS=file:/// '
    f'file:///content/hadoop_input '
    f'file:///content/hadoop_grep_output '
    f'"INFO|ERROR"'
)
run(grep_cmd, env=ENV)

run('echo "----- grep output -----" && cat /content/hadoop_grep_output/*')

print("""
Explanation:
Each line shows: <word> <count>

Example:
info 6  -> the word 'info' appears 6 times
error 2 -> the word 'error' appears 2 times

Numbers like 2026 or 04 come from timestamps in the logs.
""")

RUN: rm -rf /content/hadoop_grep_output

RUN: "/content/hadoop-3.3.6/bin/hadoop" jar "/content/hadoop-3.3.6/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.3.6.jar" grep -D mapreduce.framework.name=local -D fs.defaultFS=file:/// file:///content/hadoop_input file:///content/hadoop_grep_output "INFO|ERROR"
2026-04-23 13:00:25,065 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-04-23 13:00:25,374 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-04-23 13:00:25,374 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-04-23 13:00:25,780 INFO input.FileInputFormat: Total input files to process : 2
2026-04-23 13:00:25,834 INFO mapreduce.JobSubmitter: number of splits:2
2026-04-23 13:00:26,362 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1286867823_0001
2026-04-23 13:00:26,362 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-04-23 13:00:26,627 INFO mapreduce.Job: The url to track 

In [ ]:

# Step 5: Create mapper and reducer scripts for lowercase word count
import textwrap
from pathlib import Path

mapper_wc = textwrap.dedent("""#!/usr/bin/env python3
import sys
import re

for line in sys.stdin:
    for word in re.findall(r"[A-Za-z0-9_]+", line.lower()):
        print(f"{word}\t1")
""")

reducer_sum = textwrap.dedent("""#!/usr/bin/env python3
import sys

current_key = None
current_total = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    key, value = line.split("\t", 1)
    value = int(value)

    if key == current_key:
        current_total += value
    else:
        if current_key is not None:
            print(f"{current_key}\t{current_total}")
        current_key = key
        current_total = value

if current_key is not None:
    print(f"{current_key}\t{current_total}")
""")

Path("/content/mapper_wc.py").write_text(mapper_wc)
Path("/content/reducer_sum.py").write_text(reducer_sum)

run("chmod +x /content/mapper_wc.py /content/reducer_sum.py")
run("sed -n '1,120p' /content/mapper_wc.py")
run('echo "-----" && sed -n "1,160p" /content/reducer_sum.py')


RUN: chmod +x /content/mapper_wc.py /content/reducer_sum.py

RUN: sed -n '1,120p' /content/mapper_wc.py
#!/usr/bin/env python3
import sys
import re

for line in sys.stdin:
    for word in re.findall(r"[A-Za-z0-9_]+", line.lower()):
        print(f"{word}	1")

RUN: echo "-----" && sed -n "1,160p" /content/reducer_sum.py
-----
#!/usr/bin/env python3
import sys

current_key = None
current_total = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
    key, value = line.split("	", 1)
    value = int(value)

    if key == current_key:
        current_total += value
    else:
        if current_key is not None:
            print(f"{current_key}	{current_total}")
        current_key = key
        current_total = value

if current_key is not None:
    print(f"{current_key}	{current_total}")



CompletedProcess(args='echo "-----" && sed -n "1,160p" /content/reducer_sum.py', returncode=0, stdout='-----\n#!/usr/bin/env python3\nimport sys\n\ncurrent_key = None\ncurrent_total = 0\n\nfor line in sys.stdin:\n    line = line.strip()\n    if not line:\n        continue\n    key, value = line.split("\t", 1)\n    value = int(value)\n\n    if key == current_key:\n        current_total += value\n    else:\n        if current_key is not None:\n            print(f"{current_key}\t{current_total}")\n        current_key = key\n        current_total = value\n\nif current_key is not None:\n    print(f"{current_key}\t{current_total}")\n')

In [ ]:

# Step 6: Run Hadoop Streaming word count in LOCAL mode
run("rm -rf /content/hadoop_wc_output")

wc_cmd = (
    f'"{os.environ["HADOOP_HOME"]}/bin/hadoop" jar "{STREAMING_JAR}" '
    f'-D mapreduce.framework.name=local '
    f'-D fs.defaultFS=file:/// '
    f'-files /content/mapper_wc.py,/content/reducer_sum.py '
    f'-mapper "python3 mapper_wc.py" '
    f'-reducer "python3 reducer_sum.py" '
    f'-input file:///content/hadoop_input '
    f'-output file:///content/hadoop_wc_output'
)
run(wc_cmd, env=ENV)

run('echo "----- first 30 word-count results -----" && head -30 /content/hadoop_wc_output/part-00000')


RUN: rm -rf /content/hadoop_wc_output

RUN: "/content/hadoop-3.3.6/bin/hadoop" jar "/content/hadoop-3.3.6/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar" -D mapreduce.framework.name=local -D fs.defaultFS=file:/// -files /content/mapper_wc.py,/content/reducer_sum.py -mapper "python3 mapper_wc.py" -reducer "python3 reducer_sum.py" -input file:///content/hadoop_input -output file:///content/hadoop_wc_output
2026-04-23 13:00:31,183 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-04-23 13:00:31,378 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-04-23 13:00:31,378 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-04-23 13:00:31,406 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-04-23 13:00:31,732 INFO mapred.FileInputFormat: Total input files to process : 2
2026-04-23 13:00:31,755 INFO mapreduce.JobSubmitter: number of splits:2
2026-04-23 13:00:32,049 INFO mapreduce.Job

CompletedProcess(args='echo "----- first 30 word-count results -----" && head -30 /content/hadoop_wc_output/part-00000', returncode=0, stdout='----- first 30 word-count results -----\n01\t1\n02\t1\n03\t2\n04\t11\n05\t1\n09\t5\n10\t6\n15\t1\n2026\t10\n22\t10\n28\t1\n31\t1\n36\t1\n37\t2\n38\t2\n39\t1\n44\t1\n45\t1\n55\t1\n59\t1\naccepted\t1\nauthenticationservice\t1\nblockmanager\t1\nblockstatechange\t1\ncheckpoint\t1\ncheckpointcoordinator\t1\ncompleted\t2\nconfigurationvalidationservice\t1\ncreation\t1\ndatanode\t1\n')

In [ ]:

# Step 7: Quick checks on word-count output
run('echo "Total unique words:" && wc -l /content/hadoop_wc_output/part-00000')
print("""
Explanation:
wc -l counts the number of lines in the output file.

Each line = one unique word.
So this is the total number of distinct words in the dataset.
""")

run('echo "" && echo "Top words by frequency:" && sort -k2,2nr /content/hadoop_wc_output/part-00000 | head -15')


print("""
Explanation:
- sort -k2,2nr → sorts by the 2nd column (count), descending
- head -15 → shows top 15 most frequent words

This helps identify the most common patterns in logs.
""")

RUN: echo "Total unique words:" && wc -l /content/hadoop_wc_output/part-00000
Total unique words:
66 /content/hadoop_wc_output/part-00000


Explanation:
wc -l counts the number of lines in the output file.

Each line = one unique word.
So this is the total number of distinct words in the dataset.

RUN: echo "" && echo "Top words by frequency:" && sort -k2,2nr /content/hadoop_wc_output/part-00000 | head -15

Top words by frequency:
04	11
2026	10
22	10
10	6
info	6
09	5
03	2
37	2
38	2
completed	2
error	2
from	2
metadata	2
successfully	2
synchronization	2


Explanation:
- sort -k2,2nr → sorts by the 2nd column (count), descending
- head -15 → shows top 15 most frequent words

This helps identify the most common patterns in logs.



In [ ]:
# Step 8: Create mapper and reducer for distinct words
mapper_distinct = textwrap.dedent("""#!/usr/bin/env python3
import sys
import re

for line in sys.stdin:
    for word in re.findall(r"[A-Za-z0-9_]+", line.lower()):
        print(word)
""")

reducer_distinct = textwrap.dedent("""#!/usr/bin/env python3
import sys

previous = None

for line in sys.stdin:
    word = line.strip()
    if not word:
        continue
    if word != previous:
        print(word)
        previous = word
""")

Path("/content/mapper_distinct.py").write_text(mapper_distinct)
Path("/content/reducer_distinct.py").write_text(reducer_distinct)

run("chmod +x /content/mapper_distinct.py /content/reducer_distinct.py")


RUN: chmod +x /content/mapper_distinct.py /content/reducer_distinct.py



CompletedProcess(args='chmod +x /content/mapper_distinct.py /content/reducer_distinct.py', returncode=0, stdout='')

In [ ]:

# Step 9: Run Hadoop Streaming distinct-word job
run("rm -rf /content/hadoop_distinct_output")

distinct_cmd = (
    f'"{os.environ["HADOOP_HOME"]}/bin/hadoop" jar "{STREAMING_JAR}" '
    f'-D mapreduce.framework.name=local '
    f'-D fs.defaultFS=file:/// '
    f'-files /content/mapper_distinct.py,/content/reducer_distinct.py '
    f'-mapper "python3 mapper_distinct.py" '
    f'-reducer "python3 reducer_distinct.py" '
    f'-input file:///content/hadoop_input '
    f'-output file:///content/hadoop_distinct_output'
)
run(distinct_cmd, env=ENV)

run('echo "----- first 30 distinct words -----" && head -30 /content/hadoop_distinct_output/part-00000')

print("""
Explanation:
This output shows unique words only (no counts).

Each word appears once because duplicates are removed.
""")

run('echo "" && echo "Distinct word count:" && wc -l /content/hadoop_distinct_output/part-00000')


RUN: rm -rf /content/hadoop_distinct_output

RUN: "/content/hadoop-3.3.6/bin/hadoop" jar "/content/hadoop-3.3.6/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar" -D mapreduce.framework.name=local -D fs.defaultFS=file:/// -files /content/mapper_distinct.py,/content/reducer_distinct.py -mapper "python3 mapper_distinct.py" -reducer "python3 reducer_distinct.py" -input file:///content/hadoop_input -output file:///content/hadoop_distinct_output
2026-04-23 13:00:37,240 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-04-23 13:00:37,439 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-04-23 13:00:37,439 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-04-23 13:00:37,485 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-04-23 13:00:37,822 INFO mapred.FileInputFormat: Total input files to process : 2
2026-04-23 13:00:37,859 INFO mapreduce.JobSubmitter: number of splits:2
2026-04-

CompletedProcess(args='echo "" && echo "Distinct word count:" && wc -l /content/hadoop_distinct_output/part-00000', returncode=0, stdout='\nDistinct word count:\n66 /content/hadoop_distinct_output/part-00000\n')

In [ ]:

# Step 10: Create mapper for long words (> 12 characters)
mapper_long = textwrap.dedent("""#!/usr/bin/env python3
import sys
import re

for line in sys.stdin:
    for word in re.findall(r"[A-Za-z0-9_]+", line.lower()):
        if len(word) > 12:
            print(f"{word}\t1")
""")

Path("/content/mapper_long.py").write_text(mapper_long)
run("chmod +x /content/mapper_long.py")
run("sed -n '1,120p' /content/mapper_long.py")


RUN: chmod +x /content/mapper_long.py

RUN: sed -n '1,120p' /content/mapper_long.py
#!/usr/bin/env python3
import sys
import re

for line in sys.stdin:
    for word in re.findall(r"[A-Za-z0-9_]+", line.lower()):
        if len(word) > 12:
            print(f"{word}	1")



CompletedProcess(args="sed -n '1,120p' /content/mapper_long.py", returncode=0, stdout='#!/usr/bin/env python3\nimport sys\nimport re\n\nfor line in sys.stdin:\n    for word in re.findall(r"[A-Za-z0-9_]+", line.lower()):\n        if len(word) > 12:\n            print(f"{word}\t1")\n')

In [ ]:

# Step 11: Run Hadoop Streaming long-word job
run("rm -rf /content/hadoop_longwords_output")

long_cmd = (
    f'"{os.environ["HADOOP_HOME"]}/bin/hadoop" jar "{STREAMING_JAR}" '
    f'-D mapreduce.framework.name=local '
    f'-D fs.defaultFS=file:/// '
    f'-files /content/mapper_long.py,/content/reducer_sum.py '
    f'-mapper "python3 mapper_long.py" '
    f'-reducer "python3 reducer_sum.py" '
    f'-input file:///content/hadoop_input '
    f'-output file:///content/hadoop_longwords_output'
)
run(long_cmd, env=ENV)

run('echo "----- long words and their counts -----" && cat /content/hadoop_longwords_output/part-00000')

print("""
Explanation:
This shows words longer than 12 characters.

Example:
configurationvalidationservice → long system-related term
authenticationservice → service name

These are typically technical or system identifiers in logs.
""")

RUN: rm -rf /content/hadoop_longwords_output

RUN: "/content/hadoop-3.3.6/bin/hadoop" jar "/content/hadoop-3.3.6/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar" -D mapreduce.framework.name=local -D fs.defaultFS=file:/// -files /content/mapper_long.py,/content/reducer_sum.py -mapper "python3 mapper_long.py" -reducer "python3 reducer_sum.py" -input file:///content/hadoop_input -output file:///content/hadoop_longwords_output
2026-04-23 13:00:41,949 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-04-23 13:00:42,169 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-04-23 13:00:42,169 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-04-23 13:00:42,198 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-04-23 13:00:42,552 INFO mapred.FileInputFormat: Total input files to process : 2
2026-04-23 13:00:42,578 INFO mapreduce.JobSubmitter: number of splits:2
2026-04-23 13:00:42,867 

In [ ]:
print("""
===== FINAL SUMMARY =====

We performed:

1. Word Count
   → frequency of each word

2. Distinct Words
   → unique vocabulary size

3. Long Word Filter
   → extracted technical/system terms

This simulates real-world log analysis using Hadoop Streaming.

Key takeaway:
Hadoop processes large-scale text using MapReduce (mapper + reducer),
even though here we ran it in local mode.
""")


===== FINAL SUMMARY =====

We performed:

1. Word Count
   → frequency of each word

2. Distinct Words
   → unique vocabulary size

3. Long Word Filter
   → extracted technical/system terms

This simulates real-world log analysis using Hadoop Streaming.

Key takeaway:
Hadoop processes large-scale text using MapReduce (mapper + reducer),
even though here we ran it in local mode.

